## Interval Analysis

In [6]:
# !pip install tensorboardX
# !pip install bound-propagation

from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = False
#device = torch.device("cuda" if use_cuda else "cpu")
device = torch.device("cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
bound_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

class Net(nn.Sequential):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 50)
        self.out = nn.Linear(50, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.out(x)
        return x


stan_model = Net().to(device)
stan_model.train()


Net(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (out): Linear(in_features=50, out_features=10, bias=True)
)

In [ ]:
def train_model(model, num_epochs):
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')
    

In [5]:
startq = time.time()
train_model(stan_model, 20)
endq = time.time()

print(f"Standard Training Time: {endq - startq} seconds")

Epoch 1/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[ 3.5002e+00, -5.7343e+00,  4.9217e+00, -7.0153e+00,  2.8521e+00,
          1.1329e+00,  1.4782e+01, -6.1959e+00, -6.4175e-01, -7.8219e+00],
        [ 9.0531e+00, -5.7039e+00,  2.5185e+00, -9.2835e-01, -1.0883e+01,
          5.8338e+00, -4.4790e-01, -2.3707e+00,  3.6181e+00, -2.2712e+00],
        [-3.1649e+00,  6.1360e+00,  2.6338e+00, -1.2534e+00, -2.8729e+00,
          7.9285e-02,  1.8301e+00, -1.9887e-01,  3.6824e+00, -3.1582e+00],
        [ 5.9612e+00, -7.0752e+00, -2.7562e+00,  1.7988e+00, -6.5287e+00,
          1.2979e+01,  2.4008e+00, -1.0199e+01,  2.8304e+00, -9.6021e-01],
        [-7.5943e+00,  1.0102e+01,  3.3238e+00,  4.1009e+00, -4.0512e+00,
          7.1675e-01, -2.7406e+00,  5.1235e-01,  3.1200e+00, -2.7263e+00],
        [ 1.9243e+01, -9.6213e+00,  8.1646e+00, -6.2615e+00, -2.0324e+01,
          9.1186e+00,  3.9841e+00, -5.9167e+00,  5.6110e+00, -7.1844e+00],
        [-5.9295e+00, -4.6584e+00, -3.3157e+00,  6.2256e+00,  1.6278e+00,
          4.0748e+00, -1.3804e+0

Epoch 2/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[-5.0182e+00, -6.1691e+00, -8.5722e+00, -2.8447e-01,  7.3501e+00,
          5.9552e+00, -5.6788e+00, -2.0422e+00,  8.3908e+00,  1.1224e+01],
        [ 1.4442e-01, -4.2998e+00, -6.6170e-01, -2.0361e+00,  1.9796e+00,
         -2.4480e+00, -5.1920e+00,  9.1833e+00,  1.5926e+00,  2.8965e+00],
        [ 1.3165e+00, -7.3567e+00,  1.2179e+01,  3.7444e+00,  3.0409e-01,
         -7.3817e+00,  4.2293e+00,  2.1595e+00, -2.7127e+00, -7.1006e+00],
        [-2.0347e+00,  2.7407e+00,  4.6049e+00,  1.2440e+01, -1.0991e+01,
          4.3417e+00, -5.4429e+00, -5.5856e+00,  6.0269e+00, -4.9450e+00],
        [-9.3489e+00,  1.1591e+01,  1.6804e+00,  2.6879e+00, -2.1451e+00,
         -1.7648e+00, -4.4272e+00,  3.9679e+00,  3.2944e+00,  3.9497e-01],
        [-6.8072e+00,  1.0182e+01,  8.8868e-01,  1.0388e+00, -2.9817e+00,
          2.1408e+00,  7.2668e-01, -9.8097e-01,  4.2432e+00, -3.3694e+00],
        [-6.9121e+00,  9.8430e+00,  2.8272e+00,  3.2276e+00, -3.9911e+00,
          1.2139e+00, -1.5149e+0

Epoch 3/20:   0%|          | 0/938 [00:00<?, ?it/s]


...

tensor([[-9.1845e+00,  8.8279e+00,  2.3051e+00,  3.4496e+00, -1.7609e-01,
         -2.7272e+00, -4.9299e+00,  3.4433e+00,  4.2642e+00,  1.0025e+00],
        [ 5.6608e-01, -4.3052e+00,  6.5757e+00,  3.6730e+00, -5.7167e+00,
         -4.2980e+00, -1.0802e+01,  1.2122e+01, -4.0897e-01,  1.8917e+00],
        [-1.4482e+00, -1.9970e+00, -1.4549e+00,  9.9150e+00, -1.2886e+01,
          8.3127e+00, -1.8146e+01,  4.0663e+00,  5.5719e+00,  8.2908e+00],
        [-8.7360e-01, -1.1008e+01, -1.2330e+00,  1.8946e+00,  4.1198e+00,
          2.5097e+00, -9.6235e+00,  1.1960e+00,  3.0104e+00,  1.1743e+01],
        [ 4.2359e+00, -7.1926e+00,  4.9556e+00, -9.5447e+00,  4.5817e+00,
         -1.6854e+00,  1.5936e+01, -4.4762e+00,  2.4906e-01, -6.6331e+00],
        [-8.0090e-02, -1.8186e+00, -3.2235e-01,  3.7963e-01, -2.5219e+00,
          8.8670e+00,  8.9968e-02, -7.6133e+00,  5.3536e+00,  6.7551e-02],
        [ 1.7833e+00, -3.1494e+00, -1.1003e+00,  1.0219e+00, -2.0233e+00,
          3.2178e+00, -2.6041e+0

Epoch 19/20:   0%|          | 0/938 [00:00<?, ?it/s]


tensor([[-5.9967e+00,  7.5221e+00,  1.1533e+00,  1.5237e+00, -1.0194e+00,
         -1.0983e+00, -2.1442e+00,  2.4331e+00,  1.7347e+00, -3.8107e-01],
        [ 7.3266e-01,  3.1762e-02, -2.2480e+00,  1.6032e+00, -3.2224e+00,
          9.0613e+00, -4.8928e-01, -3.7350e+00,  6.2364e-01, -1.9878e+00],
        [ 3.4882e+00, -5.1686e+00,  2.5412e+00, -5.7838e+00,  1.8977e+00,
          2.5427e+00,  1.3555e+01, -6.9264e+00,  6.3400e-01, -7.1519e+00],
        [-1.0617e+00, -1.8717e+00,  2.4825e-02,  8.8396e+00, -5.5488e+00,
          4.6766e+00, -7.8944e+00, -5.9120e-01,  2.4457e-01,  1.5320e+00],
        [-9.4025e-01, -4.4733e+00, -1.1364e+00, -1.3712e+00,  5.5701e+00,
          2.2670e-01,  3.6914e-01, -2.4944e-01,  6.4711e-01,  2.7030e+00],
        [-2.8646e+00, -6.1472e+00, -2.5558e+00, -2.8748e+00,  1.1431e+01,
          2.3336e-01,  5.9271e-01,  1.1449e-01, -5.3506e-02,  4.9260e+00],
        [-4.0228e+00, -2.6162e+00, -1.4798e+00, -4.2951e-01,  7.6027e+00,
         -5.3280e-01,  7.8132e-0

Epoch 20/20:   0%|          | 0/938 [00:00<?, ?it/s]

tensor([[-5.0166e+00, -4.4431e+00, -4.7324e+00,  4.6374e+00,  2.1556e+00,
          1.3717e+00, -1.2486e+01,  3.5181e+00,  4.6396e+00,  1.2766e+01],
        [ 2.3661e+00, -4.2028e+00,  6.1463e+00, -7.4028e+00,  4.1042e+00,
         -5.9757e-01,  1.5524e+01, -3.8925e+00, -2.5267e+00, -9.0084e+00],
        [ 1.1883e+00,  1.3312e+00,  4.8093e+00, -2.4536e+00, -3.1505e+00,
          2.9755e+00,  1.0928e+01, -5.4234e+00,  1.1339e-01, -1.0075e+01],
        [ 3.3619e+00,  2.9867e-01,  7.2697e-01,  6.0110e-01, -9.2667e+00,
          1.0733e+01,  1.3457e+00, -7.3440e+00,  2.7867e+00, -3.3455e+00],
        [-2.4388e+00,  3.2962e-01,  1.2728e+01,  4.9048e+00, -7.3818e+00,
         -4.4657e+00, -3.7025e+00,  6.8817e-01,  4.9681e+00, -2.2511e+00],
        [-4.7714e+00, -5.0246e+00, -1.6813e+00, -2.7095e+00,  1.0967e+01,
         -1.8610e-01, -7.7679e-01,  8.3188e-01,  1.5719e+00,  6.0364e+00],
        [-5.2315e+00, -2.3498e+00, -5.3660e+00, -2.9185e+00,  1.0930e+01,
          8.5470e-01,  3.5562e-0

In [8]:
test_model(stan_model)

Accuracy on images: 96.7


In [9]:
from bound_propagation import BoundModelFactory, HyperRectangle
def rb_worst(model, x, y, eps):
    x = x.view(-1, 28*28)
    L = torch.clamp(x - eps, 0, 1)
    U = torch.clamp(x + eps, 0, 1)
    factory = BoundModelFactory()
    box = HyperRectangle(L, U)

    alter = nn.Sequential(
        model.fc1,
        nn.ReLU(),
        model.fc2,
        nn.ReLU(),
        model.fc3,
        nn.ReLU(),
        model.out   
    )

    bound_net = factory.build(alter)
    interval = bound_net.ibp(box)
    lower, higher = interval.lower, interval.upper
    mask = torch.ones_like(lower, dtype=torch.bool)
    for i in range(x.size(0)):
        mask[i, y[i]] = False
    res = torch.where(mask, higher, lower)
    loss = F.cross_entropy(res, y)
    return loss

In [10]:
def train_model_ibp(model, num_epochs):
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        kappa = 1.0 - 0.5 * (epoch / num_epochs)
        eps = 0.1 * epoch / num_epochs
        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}"):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = model(images)
            ce_loss = F.cross_entropy(logits, labels)
            rb_loss = rb_worst(model, images, labels, eps)
            loss = kappa * ce_loss + (1 - kappa) * rb_loss
            loss.backward()
            optimizer.step()
        model.eval()
        tot_val, tot_acc = 0.0, 0.0
        val_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            with torch.no_grad():
                logits = model(images)
                ce_loss = F.cross_entropy(logits, labels)
                rb_loss = rb_worst(model, images, labels, eps)
                loss = kappa * ce_loss + (1 - kappa) * rb_loss
                val_loss += loss.item()
                tot_acc += (logits.argmax(dim=1) == labels).sum().item()
                tot_val += labels.size(0)
        val_acc = 100.0 * tot_acc / tot_val

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {val_loss/len(train_loader):.3f}, Accuracy: {val_acc:.2f}%')

In [11]:
ibp_model = Net().to(device)
ibp_model.train()
start = time.time()
train_model_ibp(ibp_model, 15)
end = time.time()

print(f"IBP Training Time: {end - start} seconds")


Epoch 1/15: 100%|██████████| 938/938 [00:04<00:00, 209.65it/s]


Epoch 1/15, Loss: 0.558, Accuracy: 83.54%


Epoch 2/15: 100%|██████████| 938/938 [00:04<00:00, 214.67it/s]


Epoch 2/15, Loss: 0.455, Accuracy: 89.54%


Epoch 3/15: 100%|██████████| 938/938 [00:04<00:00, 214.93it/s]


...

Epoch 13/15, Loss: 0.891, Accuracy: 95.63%


Epoch 14/15: 100%|██████████| 938/938 [00:04<00:00, 216.44it/s]


Epoch 14/15, Loss: 0.937, Accuracy: 95.68%


Epoch 15/15: 100%|██████████| 938/938 [00:04<00:00, 215.52it/s]


Epoch 15/15, Loss: 0.986, Accuracy: 95.80%
IBP Training Time: 110.66348099708557 seconds


In [13]:
def pgd(model, x, labels, k=40, eps=8/255, eps_step=0.001):
    model.eval()
    ce_loss = F.cross_entropy
    adv_x = x.clone().detach()
    adv_x = torch.clamp(adv_x, 0.0, 1.0)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        logits = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(logits, labels)
        loss.backward()
        assert adv_x.grad is not None
        grad = adv_x.grad.data
        # TODO: compute the adv_x
        adv_x = adv_x.detach() + eps_step * torch.sign(grad)
        adv_x = torch.clamp(adv_x, x - eps, x + eps)
        adv_x = torch.clamp(adv_x, 0, 1).detach()
    return adv_x

def evaluate_pgd(model):
    model.eval()
    tot_acc = 0.0
    tot_test = 0.0
    tot_acc_adv = 0.0
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(-1, 28*28)
        adv_images = pgd(model, images, labels)
        outputs = model(adv_images)
        tot_acc_adv += (outputs.argmax(dim=1) == labels).sum().item()
        tot_test += labels.size(0)
        tot_acc += (model(images).argmax(dim=1) == labels).sum().item()
    return 100.0 * tot_acc / tot_test, 100.0 * tot_acc_adv / tot_test

stan_acc, stan_acc_adv = evaluate_pgd(stan_model)
print(f'Standard Model - Clean Accuracy: {stan_acc:.2f}%, PGD Accuracy: {stan_acc_adv:.2f}%')
ibp_acc, ibp_acc_adv = evaluate_pgd(ibp_model)
print(f'IBP Model - Clean Accuracy: {ibp_acc:.2f}%, PGD Accuracy: {ibp_acc_adv:.2f}%')


Standard Model - Clean Accuracy: 96.70%, PGD Accuracy: 81.87%
IBP Model - Clean Accuracy: 95.26%, PGD Accuracy: 80.39%


### Write the interval analysis for the simple model

In [ ]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

from bound_propagation import BoundModelFactory, HyperRectangle

factory = BoundModelFactory()
isinstance(stan_model, nn.Sequential)
net = factory.build(stan_model)

In [ ]:
net.eval()
epsilons = np.linspace(0.01, 0.1, 10)
always_correct = np.zeros(len(epsilons), dtype=int)
still_correct = np.zeros(len(epsilons), dtype=int)
correct = 0
total = 0

for images, labels in test_loader:
    images = images.view((-1, 28 * 28)).to(device)
    labels = labels.to(device)

    outputs = stan_model(images)
    _, predicted = torch.max(outputs.data, 1)
    predicted_correct = predicted == labels

    total += labels.size(0)
    correct += predicted_correct.sum().item()

    for idx, epsilon in enumerate(epsilons):
        input_bounds = HyperRectangle.from_eps(images, float(epsilon))
        crown_ibp_bounds = net.crown_ibp(input_bounds).concretize()
        lower, upper = crown_ibp_bounds.lower, crown_ibp_bounds.upper

        label_lower = lower.gather(1, labels.unsqueeze(1)).squeeze(1)
        label_mask = torch.zeros_like(upper, dtype=torch.bool)
        label_mask.scatter_(1, labels.unsqueeze(1), True)
        max_other_upper = upper.masked_fill(label_mask, float('-inf')).max(dim=1).values
        robust_mask = label_lower > max_other_upper
        always_correct[idx] += robust_mask.sum().item()

print(f'Standard accuracy: {correct / total*100:.2f}%')
for eps, ac in zip(epsilons, always_correct):
    print(f'eps={eps:.3f}: always_correct={ac / total * 100:.2f}%')

        
        
        
        
    


Standard accuracy: 96.70%
eps=0.010: always_correct=42.67%
eps=0.020: always_correct=18.58%
eps=0.030: always_correct=5.53%
eps=0.040: always_correct=1.23%
eps=0.050: always_correct=0.27%
eps=0.060: always_correct=0.08%
eps=0.070: always_correct=0.03%
eps=0.080: always_correct=0.00%
eps=0.090: always_correct=0.00%
eps=0.100: always_correct=0.00%
